In [ ]:
import argparse
import datetime
import glob
import os
import sys
import time
import warnings

import numpy as np
import pytorch_lightning as pl
import torch
import torchvision
from PIL import Image
from omegaconf import OmegaConf
from pytorch_lightning import seed_everything
from pytorch_lightning.trainer import Trainer
from pytorch_lightning.utilities import rank_zero_info
from torch.utils.data import random_split, DataLoader, Dataset, ConcatDataset
from pytorch_lightning.callbacks import Callback
from pytorch_lightning.utilities.distributed import rank_zero_only
from torchvision.transforms import Compose, ToTensor, Resize, Normalize

from cheff.ldm import instantiate_from_config
from cheff.machex import MaCheXDataset, MimicT2IDataset

import datetime
from omegaconf import OmegaConf
from pytorch_lightning import seed_everything
from pytorch_lightning.trainer import Trainer
from cheff.ldm import instantiate_from_config
from peft import LoraConfig, get_peft_model
from cheff.peft_modules.inject_lora import apply_lora_peft

d:\01 - Dokumente\01 - Studium\01 - StaDS Experience\01 - Kursmaterial\ADL - Applied Deep Learning\applied_dl\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ONLY TO ENABLE IMPORT OF ldm AND peft_modules ALSO FOR THE JUPYTER NOTEBOOK
# NOT NEEDED IF YOU RUN THE SCRIPTS DIRECTLY

# 1. Identify the absolute path to the directory CONTAINING 'ldm'
# Based on your traceback, this is the inner 'cheff' folder
project_root = os.path.abspath("..") # Adjust this so it points to 'applied_dl/cheff/'
inner_cheff_path = os.path.join(project_root, "cheff")

# 2. Add it to the system path
if inner_cheff_path not in sys.path:
    sys.path.append(inner_cheff_path)

# 3. Test if Python can now 'see' ldm directly
try:
    import ldm
    print("Success: 'ldm' module is now accessible!")
except ImportError:
    print("Error: Path still not correct. Check the location of the 'ldm' folder.")


try:
    import peft_modules
    print("Success: 'peft_modules' module is now accessible!")
except ImportError:
    print("Error: Path still not correct. Check the location of the 'peft_modules' folder.")

Success: 'ldm' module is now accessible!
Success: 'peft_modules' module is now accessible!


In [2]:
# ALL THE FUNCTIONS DEFINED IN THE 01_train_ldm.PY SCRIPT


def get_parser(**parser_kwargs):
    def str2bool(v):
        if isinstance(v, bool):
            return v
        if v.lower() in ("yes", "true", "t", "y", "1"):
            return True
        elif v.lower() in ("no", "false", "f", "n", "0"):
            return False
        else:
            raise argparse.ArgumentTypeError("Boolean value expected.")

    parser = argparse.ArgumentParser(**parser_kwargs)
    parser.add_argument(
        "-n",
        "--name",
        type=str,
        const=True,
        default="",
        nargs="?",
        help="postfix for logdir",
    )
    parser.add_argument(
        "-r",
        "--resume",
        type=str,
        const=True,
        default="",
        nargs="?",
        help="resume from logdir or checkpoint in logdir",
    )
    parser.add_argument(
        "-b",
        "--base",
        nargs="*",
        metavar="base_config.yaml",
        help="paths to base configs. Loaded from left-to-right. "
             "Parameters can be overwritten or added with command-line options of the form `--key value`.",
        default=list(),
    )
    parser.add_argument(
        "-t",
        "--train",
        type=str2bool,
        const=True,
        default=False,
        nargs="?",
        help="train",
    )
    parser.add_argument(
        "--no-test",
        type=str2bool,
        const=True,
        default=False,
        nargs="?",
        help="disable test",
    )
    parser.add_argument(
        "-p",
        "--project",
        help="name of new or path to existing project"
    )
    parser.add_argument(
        "-d",
        "--debug",
        type=str2bool,
        nargs="?",
        const=True,
        default=False,
        help="enable post-mortem debugging",
    )
    parser.add_argument(
        "-s",
        "--seed",
        type=int,
        default=23,
        help="seed for seed_everything",
    )
    parser.add_argument(
        "-f",
        "--postfix",
        type=str,
        default="",
        help="post-postfix for default name",
    )
    parser.add_argument(
        "-l",
        "--logdir",
        type=str,
        default="logs",
        help="directory for logging",
    )
    return parser


def nondefault_trainer_args(opt):
    parser = argparse.ArgumentParser()
    parser = Trainer.add_argparse_args(parser)
    args = parser.parse_args([])
    return sorted(k for k in vars(args) if getattr(opt, k) != getattr(args, k))


def worker_init_fn(_):
    worker_info = torch.utils.data.get_worker_info()

    dataset = worker_info.dataset
    worker_id = worker_info.id

    return np.random.seed(np.random.get_state()[1][0] + worker_id)


class DataModuleFromConfig(pl.LightningDataModule):

    def __init__(self,
                 batch_size, machex_path, test_size, num_workers, mimic=False,
                 *args, **kwargs):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers

        self.transforms = Compose([
            Resize(256),
            ToTensor(),
            Normalize(
                mean=(0.5, 0.5, 0.5),
                std=(0.5, 0.5, 0.5)
            )
        ])

        if not mimic:
            self.machex = MaCheXDataset(machex_path, self.transforms)
        else:
            self.machex = MimicT2IDataset(machex_path, self.transforms)

        train_size = len(self.machex) - test_size
        self.train_dataset, self.test_dataset = random_split(
            self.machex,
            (train_size, test_size),
            generator=torch.Generator().manual_seed(1337)
        )

    def train_dataloader(self):
        loader = DataLoader(
            dataset=self.train_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            pin_memory=True,
            shuffle=True
        )
        return loader

    def val_dataloader(self):
        loader = DataLoader(
            dataset=self.test_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            pin_memory=True,
            shuffle=False
        )
        return loader


class SetupCallback(Callback):
    def __init__(self, resume, now, logdir, ckptdir, cfgdir, config, lightning_config):
        super().__init__()
        self.resume = resume
        self.now = now
        self.logdir = logdir
        self.ckptdir = ckptdir
        self.cfgdir = cfgdir
        self.config = config
        self.lightning_config = lightning_config

    def on_keyboard_interrupt(self, trainer, pl_module):
        if trainer.global_rank == 0:
            print("Summoning checkpoint.")
            ckpt_path = os.path.join(self.ckptdir, "last.ckpt")
            trainer.save_checkpoint(ckpt_path)

    def on_pretrain_routine_start(self, trainer, pl_module):
        if trainer.global_rank == 0:
            # Create logdirs and save configs
            os.makedirs(self.logdir, exist_ok=True)
            os.makedirs(self.ckptdir, exist_ok=True)
            os.makedirs(self.cfgdir, exist_ok=True)

            if "callbacks" in self.lightning_config:
                if 'metrics_over_trainsteps_checkpoint' in self.lightning_config[
                    'callbacks']:
                    os.makedirs(os.path.join(self.ckptdir, 'trainstep_checkpoints'),
                                exist_ok=True)
            print("Project config")
            print(OmegaConf.to_yaml(self.config))
            OmegaConf.save(self.config,
                           os.path.join(self.cfgdir,
                                        "{}-project.yaml".format(self.now)))

            print("Lightning config")
            print(OmegaConf.to_yaml(self.lightning_config))
            OmegaConf.save(OmegaConf.create({"lightning": self.lightning_config}),
                           os.path.join(self.cfgdir,
                                        "{}-lightning.yaml".format(self.now)))

        else:
            # ModelCheckpoint callback created log directory --- remove it
            if not self.resume and os.path.exists(self.logdir):
                dst, name = os.path.split(self.logdir)
                dst = os.path.join(dst, "child_runs", name)
                os.makedirs(os.path.split(dst)[0], exist_ok=True)
                try:
                    os.rename(self.logdir, dst)
                except FileNotFoundError:
                    pass


class ImageLogger(Callback):
    def __init__(self, batch_frequency, max_images, clamp=True, increase_log_steps=True,
                 rescale=True, disabled=False, log_on_batch_idx=False,
                 log_first_step=False,
                 log_images_kwargs=None):
        super().__init__()
        self.rescale = rescale
        self.batch_freq = batch_frequency
        self.max_images = max_images
        self.logger_log_images = {
            pl.loggers.TestTubeLogger: self._testtube,
        }
        self.log_steps = [2 ** n for n in range(int(np.log2(self.batch_freq)) + 1)]
        if not increase_log_steps:
            self.log_steps = [self.batch_freq]
        self.clamp = clamp
        self.disabled = disabled
        self.log_on_batch_idx = log_on_batch_idx
        self.log_images_kwargs = log_images_kwargs if log_images_kwargs else {}
        self.log_first_step = log_first_step

    @rank_zero_only
    def _testtube(self, pl_module, images, batch_idx, split):
        for k in images:
            grid = torchvision.utils.make_grid(images[k])
            grid = (grid + 1.0) / 2.0  # -1,1 -> 0,1; c,h,w

            tag = f"{split}/{k}"
            pl_module.logger.experiment.add_image(
                tag, grid,
                global_step=pl_module.global_step)

    @rank_zero_only
    def log_local(self, save_dir, split, images,
                  global_step, current_epoch, batch_idx):
        root = os.path.join(save_dir, "images", split)
        for k in images:
            grid = torchvision.utils.make_grid(images[k], nrow=4)
            if self.rescale:
                grid = (grid + 1.0) / 2.0  # -1,1 -> 0,1; c,h,w
            grid = grid.transpose(0, 1).transpose(1, 2).squeeze(-1)
            grid = grid.numpy()
            grid = (grid * 255).astype(np.uint8)
            filename = "{}_gs-{:06}_e-{:06}_b-{:06}.png".format(
                k,
                global_step,
                current_epoch,
                batch_idx)
            path = os.path.join(root, filename)
            os.makedirs(os.path.split(path)[0], exist_ok=True)
            Image.fromarray(grid).save(path)

    def log_img(self, pl_module, batch, batch_idx, split="train"):
        check_idx = batch_idx if self.log_on_batch_idx else pl_module.global_step
        if (self.check_frequency(check_idx) and  # batch_idx % self.batch_freq == 0
                hasattr(pl_module, "log_images") and
                callable(pl_module.log_images) and
                self.max_images > 0):
            logger = type(pl_module.logger)

            is_train = pl_module.training
            if is_train:
                pl_module.eval()

            with torch.no_grad():
                images = pl_module.log_images(batch, split=split,
                                              **self.log_images_kwargs)

            for k in images:
                N = min(images[k].shape[0], self.max_images)
                images[k] = images[k][:N]
                if isinstance(images[k], torch.Tensor):
                    images[k] = images[k].detach().cpu()
                    if self.clamp:
                        images[k] = torch.clamp(images[k], -1., 1.)

            self.log_local(pl_module.logger.save_dir, split, images,
                           pl_module.global_step, pl_module.current_epoch, batch_idx)

            # wb_logger = pl.loggers.WandbLogger()
            # wb_logger.log_image(key="img/reconstruction", images=[grid])

            logger_log_images = self.logger_log_images.get(logger,
                                                           lambda *args, **kwargs: None)
            logger_log_images(pl_module, images, pl_module.global_step, split)

            if is_train:
                pl_module.train()

    def check_frequency(self, check_idx):
        if ((check_idx % self.batch_freq) == 0 or (check_idx in self.log_steps)) and (
                check_idx > 0 or self.log_first_step):
            try:
                self.log_steps.pop(0)
            except IndexError as e:
                print(e)
                pass
            return True
        return False

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        if not self.disabled and (pl_module.global_step > 0 or self.log_first_step):
            self.log_img(pl_module, batch, batch_idx, split="train")

    def on_validation_batch_end(self, trainer, pl_module, outputs, batch, batch_idx,
                                dataloader_idx):
        if not self.disabled and pl_module.global_step > 0:
            self.log_img(pl_module, batch, batch_idx, split="val")
        if hasattr(pl_module, 'calibrate_grad_norm'):
            if (
                    pl_module.calibrate_grad_norm and batch_idx % 25 == 0) and batch_idx > 0:
                self.log_gradients(trainer, pl_module, batch_idx=batch_idx)


class CUDACallback(Callback):
    # see https://github.com/SeanNaren/minGPT/blob/master/mingpt/callback.py
    def on_train_epoch_start(self, trainer, pl_module):
        # Reset the memory use counter
        torch.cuda.reset_peak_memory_stats(trainer.root_gpu)
        torch.cuda.synchronize(trainer.root_gpu)
        self.start_time = time.time()

    def on_train_epoch_end(self, trainer, pl_module, *args, **kwargs):
        torch.cuda.synchronize(trainer.root_gpu)
        max_memory = torch.cuda.max_memory_allocated(trainer.root_gpu) / 2 ** 20
        epoch_time = time.time() - self.start_time

        try:
            max_memory = trainer.training_type_plugin.reduce(max_memory)
            epoch_time = trainer.training_type_plugin.reduce(epoch_time)

            rank_zero_info(f"Average Epoch time: {epoch_time:.2f} seconds")
            rank_zero_info(f"Average Peak memory {max_memory:.2f}MiB")
        except AttributeError:
            pass


In [5]:
# 1. Setup basic variables
now = datetime.datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
parser = get_parser()
parser = Trainer.add_argparse_args(parser)

# 2. Parse with empty list to avoid Jupyter flag conflicts
opt, unknown = parser.parse_known_args(args=[])

# 3. MANUAL OVERRIDE: Point to your config file
# Without this, 'opt.base' is an empty list, and the config will be empty
opt.base = ["configs/toy_config.yml"] # Ensure this file exists

# 4. Initialize and merge configs safely
configs = [OmegaConf.load(f"../{cfg}") for cfg in opt.base]
cli = OmegaConf.from_dotlist(unknown)
config = OmegaConf.merge(*configs, cli)

# 5. Use .get() instead of .pop() for Jupyter
# This allows you to re-run this cell without losing the 'lightning' key
lightning_config = config.get("lightning", OmegaConf.create())
trainer_config = lightning_config.get("trainer", OmegaConf.create())

# 6. Notebook-specific trainer adjustments
trainer_config['strategy'] = None  # DDP usually crashes in interactive kernels
trainer_config['max_epochs'] = -1

for k in nondefault_trainer_args(opt):
    trainer_config[k] = getattr(opt, k)

trainer_opt = argparse.Namespace(**trainer_config)

config.model.params.first_stage_config.params.ckpt_path = None
# 7. Instantiate the model
# Now 'config.model' will exist because we manually set opt.base
lora_config_dict = config.model.params.pop("lora_config", None)

model = instantiate_from_config(config.model)

print("Cheff model successfully instantiated in Notebook!")

LatentDiffusion: Running in eps-prediction mode
DiffusionWrapper has 1.62 M params.
Keeping EMAs of 210.
making attention of type 'vanilla' with 64 in_channels
Working with z of shape (1, 3, 128, 128) = 49152 dimensions.
making attention of type 'vanilla' with 64 in_channels


d:\01 - Dokumente\01 - Studium\01 - StaDS Experience\01 - Kursmaterial\ADL - Applied Deep Learning\applied_dl\.venv\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Cheff model successfully instantiated in Notebook!


In [ ]:
# WHAT TO ADD FOR LORA PEFT

#lora_config_dict = config.model.params.pop("lora_config", None)

if config.model.params.cond_stage_trainable == True:
    print("Warning: cond_stage_trainable is set to True. Thus, BERT encoder will be included in training.")

# Actual Implementation

if lora_config_dict and lora_config_dict.enabled:
    print(f"Applying LoRA PEFT | Scope: {lora_config_dict.adaptation_scope} | Rank: {lora_config_dict.rank}")
    
    model = apply_lora_peft(model, lora_config_dict)

else:
    print("LoRA is disabled in the YAML config. Proceeding with full weights.")


Applying LoRA PEFT | Scope: attn | Rank: 16
--- PEFT Status ---
trainable params: 221184 || all params: 1845027 || trainable%: 11.988117247064677


In [2]:
def finalize_lora_training(model):
    # 1. First, freeze everything
    for param in model.parameters():
        param.requires_grad = False
    
    # 2. Then, unfreeze only the parameters added by LoRA
    # This identifies the low-rank A and B matrices you've injected
    trainable_params = 0
    for name, param in model.named_parameters():
        if "lora_" in name:
            param.requires_grad = True
            trainable_params += param.numel()
            
    print(f"LoRA setup complete. Trainable parameters: {trainable_params:,}")
    return model

def check_trainable_parameters(model):
    print("\n--- Trainable Layers Check ---")
    trainable_count = 0
    all_count = 0
    
    for name, param in model.named_parameters():
        all_count += param.numel()
        if param.requires_grad:
            print(f"TRAINABLE: {name} | Shape: {list(param.shape)}")
            trainable_count += param.numel()
            
    print("-" * 30)
    print(f"Total Parameters: {all_count:,}")
    print(f"Trainable Parameters: {trainable_count:,}")
    print(f"Percentage Trainable: {100 * trainable_count / all_count:.4f}%")
    print("-" * 30 + "\n")

In [ ]:
    # trainer and callbacks
    trainer_kwargs = dict()

    # default logger configs
    default_logger_cfgs = {
        'wandb': {
            'target': 'pytorch_lightning.loggers.WandbLogger',
            'params': {
                'project': 'cheff',
                'save_dir': logdir,
                'offline': opt.debug,
            }
        },
        'testtube': {
            'target': 'pytorch_lightning.loggers.TestTubeLogger',
            'params': {
                'name': 'testtube',
                'save_dir': logdir,
            }
        },
    }
    default_logger_cfg = default_logger_cfgs['wandb']
    if 'logger' in lightning_config:
        logger_cfg = lightning_config.logger

    trainer_kwargs['logger'] = instantiate_from_config(logger_cfg)

    # modelcheckpoint - use TrainResult/EvalResult(checkpoint_on=metric) to
    # specify which metric is used to determine best models
    default_modelckpt_cfg = {
        'target': 'pytorch_lightning.callbacks.ModelCheckpoint',
        'params': {
            'dirpath': ckptdir,
            'filename': '{epoch:06}',
            'verbose': True,
            'save_last': True,
        }
    }
    if hasattr(model, "monitor"):
        print(f"Monitoring {model.monitor} as checkpoint metric.")
        default_modelckpt_cfg["params"]["monitor"] = model.monitor
        default_modelckpt_cfg["params"]["save_top_k"] = 3

    if "modelcheckpoint" in lightning_config:
        modelckpt_cfg = lightning_config.modelcheckpoint
    else:
        modelckpt_cfg = OmegaConf.create()
    modelckpt_cfg = OmegaConf.merge(default_modelckpt_cfg, modelckpt_cfg)
    print(f"Merged modelckpt-cfg: \n{modelckpt_cfg}")

    # add callback which sets up log directory
    default_callbacks_cfg = {
        "setup_callback": {
            "target": "scripts.01_train_ldm.SetupCallback",
            "params": {
                "resume": opt.resume,
                "now": now,
                "logdir": logdir,
                "ckptdir": ckptdir,
                "cfgdir": cfgdir,
                "config": config,
                "lightning_config": lightning_config,
            }
        },
        "image_logger": {
            "target": "scripts.01_train_ldm.ImageLogger",
            "params": {
                "batch_frequency": 750,
                "max_images": 4,
                "clamp": True
            }
        },
        "cuda_callback": {
            "target": "scripts.01_train_ldm.CUDACallback"
        },

        'checkpoint_callback': modelckpt_cfg,
    }

    if "callbacks" in lightning_config:
        callbacks_cfg = lightning_config.callbacks
    else:
        callbacks_cfg = OmegaConf.create()

    callbacks_cfg = OmegaConf.merge(default_callbacks_cfg, callbacks_cfg)
    if 'ignore_keys_callback' in callbacks_cfg and hasattr(trainer_opt,
                                                            'resume_from_checkpoint'):
        callbacks_cfg.ignore_keys_callback.params[
            'ckpt_path'] = trainer_opt.resume_from_checkpoint
    elif 'ignore_keys_callback' in callbacks_cfg:
        del callbacks_cfg['ignore_keys_callback']

    if 'WandbLogger' not in logger_cfg['target']:
        del callbacks_cfg['image_logger']

    trainer_kwargs["callbacks"] = [instantiate_from_config(callbacks_cfg[k]) for k
                                    in callbacks_cfg]

    trainer = Trainer.from_argparse_args(trainer_opt, **trainer_kwargs)
    trainer.logdir = logdir  ###

    # data
    data = DataModuleFromConfig(**config.data.params)
    # NOTE according to https://pytorch-lightning.readthedocs.io/en/latest/datamodules.html
    # calling these ourselves should not be necessary but it is.
    # lightning still takes care of proper multiprocessing though
    data.prepare_data()
    data.setup()

    # configure learning rate
    bs, base_lr = config.data.params.batch_size, config.model.base_learning_rate
    if not cpu:
        ngpu = torch.cuda.device_count()
    else:
        ngpu = 1
    if 'accumulate_grad_batches' in lightning_config.trainer:
        accumulate_grad_batches = lightning_config.trainer.accumulate_grad_batches
    else:
        accumulate_grad_batches = 1
    print(f"accumulate_grad_batches = {accumulate_grad_batches}")
    lightning_config.trainer.accumulate_grad_batches = accumulate_grad_batches

    model.learning_rate = base_lr
    print(f"Setting learning rate to {model.learning_rate:.2e}")

    # allow checkpointing via USR1
    def melk(*args, **kwargs):
        # run all checkpoint hooks
        if trainer.global_rank == 0:
            print("Summoning checkpoint.")
            ckpt_path = os.path.join(ckptdir, "last.ckpt")
            trainer.save_checkpoint(ckpt_path)


    def divein(*args, **kwargs):
        if trainer.global_rank == 0:
            import pudb;
            pudb.set_trace()


    import signal

    signal.signal(signal.SIGUSR1, melk)
    signal.signal(signal.SIGUSR2, divein)

    # run
    if opt.train:
        try:
            trainer.fit(model, datamodule=data)
        except Exception:
            melk()
            raise
    if not opt.no_test and not trainer.interrupted:
        trainer.test(model, data)
except Exception as e:
    print(e)
    if opt.debug and trainer.global_rank == 0:
        try:
            import pudb as debugger
        except ImportError:
            import pdb as debugger
        debugger.post_mortem()
    raise
finally:
    if 'trainer' in locals():
        # move newly created debug project to debug_runs
        if opt.debug and not opt.resume and trainer.global_rank == 0:
            dst, name = os.path.split(logdir)
            dst = os.path.join(dst, "debug_runs", name)
            os.makedirs(os.path.split(dst)[0], exist_ok=True)
            os.rename(logdir, dst)
        if trainer.global_rank == 0:
            print(trainer.profiler.summary())
